### 引言

普通的 RNN 只有一个简单的 tanh 层，它像个只会死记硬背的学生，不管重不重要的信息都混在一起传下去，导致时间久了重要的信息就被“冲淡”了。

我们需要模型能够记住重要的信息，忘记不重要的信息，因此设计了“记忆”控制模块：门控循环单元（GRU）和长短期记忆（LSTM）。

### GRU

GRU 引入了两个“门”来控制信息流：
更新门 (Update Gate)：
- 作用：决定保留多少过去的记忆。
- 比喻：就像你在读小说，读到精彩的地方，你会把之前的剧情记得牢牢的（保留旧记忆）；读到无关紧要的流水账，你就准备把之前的剧情忘掉，更新成新的（更新记忆）。

重置门 (Reset Gate)：
- 作用：决定丢弃多少过去的记忆，以便计算新的候选状态。
- 比喻：当你读到“突然，画风一转...”，你会忽略之前的上下文，专注于当前的新情况。

![GRU](./PPT-figure/GRU.png)

| 特性 | 普通 RNN | GRU (门控循环单元) |
| :--- | :--- | :--- |
| 记忆机制 | 简单的循环，信息直接覆盖 | 通过更新门和重置门控制信息流 |
| 长距离依赖 | 差 (容易出现梯度消失) | 好 (能有效保留长期信息) |
| 参数量 | 少 | 较多 (约为 RNN 的 3 倍) |
| 计算速度 | 快 | 稍慢 (但比 LSTM 快) |
| 适用场景 | 极短序列 | 大多数序列任务 (文本、语音、时间序列) |



### LSTM

如果说 GRU 是“二门控制”，那么 LSTM 就是更精细的“三门控制”。它在 PyTorch 中也是最常用的序列模型之一。

LSTM 和 GRU 最大的区别在于，LSTM 拥有**两条**信息传递的通道：

1.  **细胞状态 (Cell State, $C_t$)**：
    *   这是 LSTM 的**“长期记忆”**（相当于电脑的硬盘）。
    *   它像一条高速公路，贯穿整个序列，信息可以在上面无损地流动。它负责保存那些“很久以前但很重要”的信息（比如句首的主语）。
2.  **隐藏状态 (Hidden State, $h_t$)**：
    *   这是 LSTM 的**“短期记忆”**（相当于内存/CPU寄存器）。
    *   它负责处理当前的任务，并输出给下一个时间步。

---

#### LSTM 的“三门”机制

为了控制这条“长期记忆”高速公路（细胞状态），LSTM 设计了三个精密的“阀门”（门控）：

1. 遗忘门 (Forget Gate) —— “扔掉垃圾”
*   **作用**：决定从长期记忆（$C_{t-1}$）中**丢弃**哪些信息。
*   **机制**：它读取上一时刻的隐藏状态 $h_{t-1}$ 和当前输入 $x_t$，输出一个 0 到 1 的数值。
    *   **0** 表示“完全忘记”。
    *   **1** 表示“完全保留”。
*   **例子**：如果你在读小说，主角换了名字，遗忘门就会把旧名字的記憶“关掉”（变为0）。

2. 输入门 (Input Gate) —— “写入新知”
*   **作用**：决定将哪些**新信息**存入长期记忆。
*   **机制**：这一步其实包含两个部分：
    1.  **Sigmoid 层（输入门）**：决定**“写哪里”**（哪些位置需要更新）。
    2.  **Tanh 层（候选状态 $\tilde{C}_t$）**：生成**“写什么”**（新的候选记忆内容）。这里再次用到了 **Tanh**，因为它能生成包含正负值的丰富信息。
*   **操作**：将两者相乘，把重要的新信息加入细胞状态。

3. 细胞状态更新 (Cell State Update)
*   这是 LSTM 最神奇的一步。
*   **公式逻辑**：$$C_t = (\text{遗忘门} \times C_{t-1}) + (\text{输入门} \times \tilde{C}_t)$$
*   **意义**：旧的长期记忆 $\times$ 遗忘比例 + 新的候选记忆 $\times$ 输入比例。这就是为什么 LSTM 能记住很久以前的东西——只要遗忘门一直是 1，信息就能一直传下去。

4. 输出门 (Output Gate) —— “对外输出”
*   **作用**：基于更新后的长期记忆（$C_t$），决定**输出什么**作为当前的短期记忆（$h_t$）。
*   **机制**：
    1.  先通过一个 **Tanh** 层处理 $C_t$（把值压缩到 -1 到 1，作为候选输出）。
    2.  再通过 **Sigmoid** 层（输出门）决定**“输出哪些部分”**。
*   **例子**：虽然你脑子里记得所有剧情（$C_t$），但当你回答别人问题时（$h_t$），你只输出与问题相关的部分。

![LSTM.png](./PPT-figure/LSTM.png)

如果把上面的文字转化为数学公式，LSTM 的计算过程如下：

1.  **遗忘门**：$f_t = \sigma(W_f \cdot [h_{t-1}, x_t] + b_f)$
2.  **输入门**：$i_t = \sigma(W_i \cdot [h_{t-1}, x_t] + b_i)$
3.  **候选状态**：$\tilde{C}_t = \tanh(W_C \cdot [h_{t-1}, x_t] + b_C)$
4.  **更新细胞状态**：$C_t = f_t \odot C_{t-1} + i_t \odot \tilde{C}_t$
5.  **输出门**：$o_t = \sigma(W_o \cdot [h_{t-1}, x_t] + b_o)$
6.  **输出隐藏状态**：$h_t = o_t \odot \tanh(C_t)$



| 特性 | LSTM (长短期记忆网络) | GRU (门控循环单元) |
| :--- | :--- | :--- |
| **结构复杂度** | **高** (3个门，2个状态) | **低** (2个门，1个状态) |
| **参数量** | 较多 (计算量大，训练稍慢) | 较少 (计算快，训练效率高) |
| **控制精度** | **更精细** (分离了记忆和输出) | 较粗 (记忆和输出混在一起) |
| **表现效果** | 在大数据集上通常略好 | 在小数据集上通常略好 |
| **适用场景** | 需要极高精度的任务 (如机器翻译) | 实时性要求高、数据量一般的任务 |

为什么C能保存长期记忆而H只是短期记忆
| 特性 |  $ C_t $  (细胞状态) |  $ h_t $  (隐藏状态) |
| :--- | :--- | :--- |
| 更新方式 | 加法 (线性叠加) | 乘法/非线性 (重新计算) |
| 梯度流动 | 畅通无阻 (导数为1[加法]，能传很远) | 容易受阻 (经过激活函数，易消失) |
| 内容性质 | 累积的历史信息，变化缓慢 | 当前的即时状态，变化剧烈 |
| 生命周期 | 贯穿整个序列 (长期) | 仅影响当前及下一步 (短期) |




### 💻 PyTorch 中的使用

在 PyTorch 中，使用 LSTM 和 GRU 几乎一样，只需要把类名换一下：

```python
# 定义 LSTM 层
# input_size: 输入特征维度
# hidden_size: 隐藏层维度
# num_layers: 层数
# batch_first: True 表示输入形状为 (Batch, Seq, Feature)
self.lstm = nn.LSTM(input_size=256, hidden_size=512, num_layers=1, batch_first=True)

# 前向传播
# output: 所有时间步的输出
# (h_n, c_n): 最后一个时间步的隐藏状态(h)和细胞状态(c)
output, (h_n, c_n) = self.lstm(embedded)
```

你会发现 LSTM 的输出多了一个 `c_n`（细胞状态），这就是它“长期记忆”的体现。
